# 第 5 周练习：针对 Insurellm 进行元数据过滤的 RAG
## 作者：瓦坎达团队的 Mougang Thomas Gasmyr

### 问题
标准 RAG 管道从 **所有** 文档类型中检索前 K 个最相似的块
（公司、合同、员工、产品），无论问题如何。像这样的问题
*“詹姆斯·威尔逊的薪水是多少？”* 在不相关的产品或合同块上浪费了检索时间。

### 解决方案：元数据过滤
在相似性搜索之前，我们使用 **LLM 分类器** 来检测问题的文档类型
目标，然后将“where”过滤器传递给 ChromaDB，以便只有相关的块竞争前 K 个插槽。

**管道**：问题→对文档类型进行分类→过滤+检索→重新排名→答案

### 这本笔记本涵盖了什么内容
1.连接到现有的预处理ChromaDB
2. 构建基于LLM的问题分类器
3. 通过查询重写和重新排序实现过滤检索
4. 比较过滤与未过滤的检索质量
5. 使用 MRR、nDCG 和关键词覆盖率指标进行评估
6. 部署Gradio聊天界面

In [ ]:
!pip install -q openai chromadb litellm pydantic python-dotenv tenacity tqdm gradio

In [ ]:
import os
import json
import math
from pathlib import Path
from collections import Counter

from openai import OpenAI
from dotenv import load_dotenv
from chromadb import PersistentClient
from litellm import completion
from pydantic import BaseModel, Field
from tenacity import retry, wait_exponential
import gradio as gr

load_dotenv(override=True)

# --- Constants ---
MODEL = "openai/gpt-4.1-nano"
DB_NAME = str(Path("../../week5/preprocessed_db").resolve())
TESTS_PATH = Path("../../week5/evaluation/tests.jsonl").resolve()
EMBEDDING_MODEL = "text-embedding-3-large"
COLLECTION_NAME = "docs"
DOC_TYPES = ["company", "contracts", "employees", "products"]

RETRIEVAL_K = 20
FINAL_K = 10

wait = wait_exponential(multiplier=1, min=10, max=240)
openai_client = OpenAI()

print(f"DB path: {DB_NAME}")
print(f"Tests path: {TESTS_PATH}")

## 1. 文档摄取（参考）

我们重用已经由 `week5/pro_implementation/ingest.py` 构建的 ChromaDB。
每个块都包含元数据：
```python
{"source": "knowledge-base/employees/james_wilson.md", "type": "employees"}
```

“type”字段直接映射到知识库文件夹：
- **公司** — 关于、职业、文化、概述
- **合同** — 32 份保险合作协议
- **员工** — 32 条员工 HR 记录
- **产品** — 8 种保险产品（Carllm、Bizllm 等）

### 先决条件：首先运行摄取

在运行此笔记本之前，您必须通过从存储库根运行摄取脚本来填充 ChromaDB：

```bash
cd week5 && uv run pro_implementation/ingest.py
```

这将：
1.从`knowledge-base/`加载所有76个markdown文件
2.使用LLM将它们分成带有标题和摘要的块
3. 使用“text-embedding-3-large”创建嵌入
4. 将所有内容存储在 `week5/preprocessed_db/` 中

等到您看到“摄取完成”后再继续。下面的单元格应该显示非零块计数。

In [ ]:
# Connect to the existing ChromaDB

chroma = PersistentClient(path=DB_NAME)
collection = chroma.get_or_create_collection(COLLECTION_NAME)

print(f"Collection '{COLLECTION_NAME}' has {collection.count()} chunks")

# Verify metadata types are present
sample = collection.peek(limit=5)
print(f"\nSample metadata: {sample['metadatas'][:3]}")

In [ ]:
# Inspect the type metadata distribution in ChromaDB

all_data = collection.get(include=["metadatas"])
all_metas = all_data["metadatas"]

# Count types
type_counts = Counter(m.get("type", "MISSING") for m in all_metas)
print("Type distribution in ChromaDB:")
for doc_type, count in type_counts.most_common():
    print(f"  {doc_type}: {count} chunks")

# Check for unexpected or missing types
expected = {"company", "contracts", "employees", "products"}
unexpected = {m.get("type") for m in all_metas} - expected
if unexpected:
    print(f"\nUnexpected type values: {unexpected}")
else:
    print(f"\nAll types are valid: {expected}")

# Show a few samples per type
print("\nSample sources per type:")
by_type = {}
for m in all_metas:
    t = m.get("type", "MISSING")
    if t not in by_type:
        by_type[t] = []
    by_type[t].append(m.get("source", "?"))

for t, sources in sorted(by_type.items()):
    unique_sources = sorted(set(sources))
    print(f"\n  {t} ({len(unique_sources)} unique source files, {len(sources)} chunks):")
    for s in unique_sources[:5]:
        print(f"    - {s}")
    if len(unique_sources) > 5:
        print(f"    ... and {len(unique_sources) - 5} more")

## 2.问题分类（新增部分）

关键思想：在查询向量存储之前，要求 LLM 对哪个文档进行分类
问题涉及的类型。

这通过 Pydantic 使用**结构化输出**，因此 LLM 返回一个干净的类型列表。
对于跨越多种类型的模糊问题（例如，*“哪个员工管理 Bizllm 合同？”*），
分类器可以返回多种类型。

**重要的设计选择：** 我们始终在过滤器中包含“公司”，因为公司文档
（about.md、overview.md、culture.md）包含有关员工的横切摘要统计数据，
产品、合同。如果没有这个，就会出现诸如*“Insurellm 有多少员工？”* 之类的问题。
会错过答案，因为计数位于公司概述文档中，而不是单个员工记录中。

In [ ]:
class QuestionClassification(BaseModel):
    doc_types: list[str] = Field(
        description=(
            "The document types most relevant to this question. "
            "Choose from: company, contracts, employees, products. "
            "Can include multiple types if the question spans categories."
        )
    )


CLASSIFY_PROMPT = f"""You classify user questions about the company Insurellm into document categories.
The knowledge base has exactly 4 document types:
- company: general company info (founding, vision, headquarters, culture, careers)
- contracts: insurance partnership agreements with other companies
- employees: individual employee HR records (salary, performance, title, history, awards, recognition, certifications)
- products: insurance product details (Carllm, Bizllm, Claimllm, Healthllm, Homellm, Lifellm, Markellm, Rellm) including features and pricing

IMPORTANT: Questions about awards, recognition, or individual achievements should ALWAYS include "employees",
because award details are documented in individual employee records.

Return ONLY the relevant type(s). If the question clearly targets one type, return just that one.
If it spans multiple types, return all relevant ones.
Valid values: {DOC_TYPES}"""


@retry(wait=wait)
def classify_question(question: str) -> list[str]:
    """Classify a question into one or more document types.
    Always includes 'company' since company docs contain cross-cutting
    summary stats (employee counts, contract totals, product history, etc.).
    """
    messages = [
        {"role": "system", "content": CLASSIFY_PROMPT},
        {"role": "user", "content": question},
    ]
    response = completion(
        model=MODEL,
        messages=messages,
        response_format=QuestionClassification,
    )
    result = QuestionClassification.model_validate_json(
        response.choices[0].message.content
    )
    # Filter to valid types only
    valid_types = [t for t in result.doc_types if t in DOC_TYPES]
    if not valid_types:
        return DOC_TYPES  # fallback: search all

    # Always include 'company' — company docs contain cross-cutting
    # overview data (employee counts, contract totals, product history)
    if "company" not in valid_types:
        valid_types.append("company")

    return valid_types

In [ ]:
# Test the classifier with various question types

test_questions = [
    "What is James Wilson's salary?",
    "What is Carllm's Enterprise pricing?",
    "Which employee manages the Bizllm contracts?",
    "When was Insurellm founded?",
    "What is the monthly cost of Homellm's Standard Tier?",
    "How many active contracts does Insurellm manage?",
    "Who won the prestigious IIOTY award in 2023?",
]

for q in test_questions:
    types = classify_question(q)
    print(f"Q: {q}")
    print(f"   -> {types}\n")

## 3. 过滤检索管道

核心更改：“fetch_context_unranked()”现在接受可选的“doc_types”参数。
当提供时，它会将“where”过滤器传递给 ChromaDB：
```python
collection.query(..., where={"type": {"$in": doc_types}})
```

这确保在相似性搜索期间仅考虑相关类型的块。
管道的其余部分（查询重写、重新排名、合并）保持不变。

In [ ]:
# --- Pydantic models for retrieval ---

class Result(BaseModel):
    page_content: str
    metadata: dict


class RankOrder(BaseModel):
    order: list[int] = Field(
        description="The order of relevance of chunks, from most relevant to least relevant, by chunk id number"
    )


# --- Core retrieval functions ---

def fetch_context_unranked(question: str, doc_types: list[str] | None = None) -> list[Result]:
    """Retrieve top-K chunks from ChromaDB, optionally filtered by doc type."""
    query = openai_client.embeddings.create(
        model=EMBEDDING_MODEL, input=[question]
    ).data[0].embedding

    query_params = {"query_embeddings": [query], "n_results": RETRIEVAL_K}

    # Apply metadata filter if specific types are identified
    if doc_types and len(doc_types) < len(DOC_TYPES):
        query_params["where"] = {"type": {"$in": doc_types}}

    results = collection.query(**query_params)

    chunks = []
    for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
        chunks.append(Result(page_content=doc, metadata=meta))
    return chunks


@retry(wait=wait)
def rerank(question: str, chunks: list[Result]) -> list[Result]:
    """Use LLM to rerank retrieved chunks by relevance."""
    system_prompt = (
        "You are a document re-ranker. "
        "You are provided with a question and a list of relevant chunks of text from a query of a knowledge base. "
        "You must rank order the provided chunks by relevance to the question, with the most relevant chunk first. "
        "Reply only with the list of ranked chunk ids, nothing else. Include all the chunk ids you are provided with, reranked."
    )
    user_prompt = (
        f"The user has asked the following question:\n\n{question}\n\n"
        "Order all the chunks of text by relevance to the question, from most relevant to least relevant. "
        "Include all the chunk ids you are provided with, reranked.\n\n"
        "Here are the chunks:\n\n"
    )
    for index, chunk in enumerate(chunks):
        user_prompt += f"# CHUNK ID: {index + 1}:\n\n{chunk.page_content}\n\n"
    user_prompt += "Reply only with the list of ranked chunk ids, nothing else."

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    response = completion(model=MODEL, messages=messages, response_format=RankOrder)
    order = RankOrder.model_validate_json(response.choices[0].message.content).order
    return [chunks[i - 1] for i in order if 1 <= i <= len(chunks)]


@retry(wait=wait)
def rewrite_query(question: str, history: list[dict] | None = None) -> str:
    """Rewrite the user's question for better knowledge base retrieval."""
    history = history or []
    message = f"""You are in a conversation with a user, answering questions about the company Insurellm.
You are about to look up information in a Knowledge Base to answer the user's question.

This is the history of your conversation so far with the user:
{history}

And this is the user's current question:
{question}

Respond only with a short, refined question that you will use to search the Knowledge Base.
It should be a VERY short specific question most likely to surface content. Focus on the question details.
IMPORTANT: Respond ONLY with the precise knowledgebase query, nothing else."""
    response = completion(model=MODEL, messages=[{"role": "system", "content": message}])
    return response.choices[0].message.content


def merge_chunks(chunks: list[Result], reranked: list[Result]) -> list[Result]:
    """Merge two lists of chunks, removing duplicates."""
    merged = chunks[:]
    existing = {chunk.page_content for chunk in chunks}
    for chunk in reranked:
        if chunk.page_content not in existing:
            merged.append(chunk)
            existing.add(chunk.page_content)
    return merged

In [ ]:
# --- Main pipeline with metadata filtering ---

def fetch_context_with_filter(original_question: str) -> tuple[list[Result], list[str]]:
    """Full retrieval pipeline: classify -> rewrite -> retrieve with filter -> merge -> rerank.
    Returns (top chunks, classified doc_types).
    """
    doc_types = classify_question(original_question)
    rewritten_question = rewrite_query(original_question)

    chunks1 = fetch_context_unranked(original_question, doc_types)
    chunks2 = fetch_context_unranked(rewritten_question, doc_types)
    chunks = merge_chunks(chunks1, chunks2)

    reranked = rerank(original_question, chunks)
    return reranked[:FINAL_K], doc_types


def fetch_context_no_filter(original_question: str) -> list[Result]:
    """Same pipeline but WITHOUT metadata filtering (baseline for comparison)."""
    rewritten_question = rewrite_query(original_question)

    chunks1 = fetch_context_unranked(original_question, doc_types=None)
    chunks2 = fetch_context_unranked(rewritten_question, doc_types=None)
    chunks = merge_chunks(chunks1, chunks2)

    reranked = rerank(original_question, chunks)
    return reranked[:FINAL_K]

In [ ]:
# --- Answer generation ---

SYSTEM_PROMPT = """You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
Your answer will be evaluated for accuracy, relevance and completeness, so make sure it only answers the question and fully answers it.
If you don't know the answer, say so.
For context, here are specific extracts from the Knowledge Base that might be directly relevant to the user's question:
{context}

With this context, please answer the user's question. Be accurate, relevant and complete."""


@retry(wait=wait)
def answer_question(question: str, history: list[dict] | None = None) -> tuple[str, list[Result], list[str]]:
    """Answer a question using RAG with metadata filtering.
    Returns (answer, retrieved_chunks, classified_doc_types).
    """
    history = history or []
    chunks, doc_types = fetch_context_with_filter(question)

    context = "\n\n".join(
        f"Extract from {chunk.metadata['source']}:\n{chunk.page_content}"
        for chunk in chunks
    )
    system_prompt = SYSTEM_PROMPT.format(context=context)

    messages = (
        [{"role": "system", "content": system_prompt}]
        + history
        + [{"role": "user", "content": question}]
    )
    response = completion(model=MODEL, messages=messages)
    return response.choices[0].message.content, chunks, doc_types


@retry(wait=wait)
def answer_question_no_filter(question: str, history: list[dict] | None = None) -> tuple[str, list[Result]]:
    """Answer a question using RAG WITHOUT metadata filtering (baseline)."""
    history = history or []
    chunks = fetch_context_no_filter(question)

    context = "\n\n".join(
        f"Extract from {chunk.metadata['source']}:\n{chunk.page_content}"
        for chunk in chunks
    )
    system_prompt = SYSTEM_PROMPT.format(context=context)

    messages = (
        [{"role": "system", "content": system_prompt}]
        + history
        + [{"role": "user", "content": question}]
    )
    response = completion(model=MODEL, messages=messages)
    return response.choices[0].message.content, chunks

## 4. 比较：过滤检索与未过滤检索

让我们比较一下使用和不使用元数据过滤时检索到的块。
对于我们显示的每个问题：
- 机密文件类型
- 检索到的块的文档类型（带过滤器与不带过滤器）
- 过滤器是否减少噪音（不相关的块类型）

In [ ]:
comparison_questions = [
    "What is James Wilson's salary?",
    "What is the monthly cost of Carllm's Enterprise Tier?",
    "When was Insurellm founded?",
    "How many Bizllm contracts has Insurellm secured?",
]

for q in comparison_questions:
    print(f"{'=' * 80}")
    print(f"Q: {q}\n")

    # With filter
    filtered_chunks, doc_types = fetch_context_with_filter(q)
    filtered_types = [c.metadata.get("type", "?") for c in filtered_chunks]

    # Without filter
    unfiltered_chunks = fetch_context_no_filter(q)
    unfiltered_types = [c.metadata.get("type", "?") for c in unfiltered_chunks]

    print(f"Classified as: {doc_types}")
    print(f"\nWith filter    - chunk types: {filtered_types}")
    print(f"Without filter - chunk types: {unfiltered_types}")

    # Count noise reduction
    noise_before = sum(1 for t in unfiltered_types if t not in doc_types)
    noise_after = sum(1 for t in filtered_types if t not in doc_types)
    print(f"\nIrrelevant chunks: {noise_before} (unfiltered) -> {noise_after} (filtered)")
    print()

## 5. 评估：MRR、nDCG 和关键词覆盖率

我们使用与“week5/evaluation/eval.py”相同的指标来衡量检索质量：

- **MRR**（平均倒数排名）：第一个相关结果出现的速度（1/排名）
- **nDCG**（标准化贴现累积增益）：整体排名的质量
- **关键字覆盖率**：预期关键字出现在检索到的块中的百分比是多少

我们对 150 个测试问题的样本进行了过滤与未过滤的比较。

In [ ]:
# --- Evaluation functions ---

class TestQuestion(BaseModel):
    question: str
    keywords: list[str]
    reference_answer: str
    category: str


def load_tests(path: str | None = None) -> list[TestQuestion]:
    """Load test questions from JSONL file."""
    test_file = path or str(TESTS_PATH)
    tests = []
    with open(test_file, "r", encoding="utf-8") as f:
        for line in f:
            data = json.loads(line.strip())
            tests.append(TestQuestion(**data))
    return tests


def calculate_mrr(keyword: str, retrieved_docs: list[Result]) -> float:
    """Reciprocal rank for a single keyword."""
    keyword_lower = keyword.lower()
    for rank, doc in enumerate(retrieved_docs, start=1):
        if keyword_lower in doc.page_content.lower():
            return 1.0 / rank
    return 0.0


def calculate_dcg(relevances: list[int], k: int) -> float:
    """Discounted Cumulative Gain."""
    dcg = 0.0
    for i in range(min(k, len(relevances))):
        dcg += relevances[i] / math.log2(i + 2)
    return dcg


def calculate_ndcg(keyword: str, retrieved_docs: list[Result], k: int = 10) -> float:
    """nDCG for a single keyword (binary relevance)."""
    keyword_lower = keyword.lower()
    relevances = [
        1 if keyword_lower in doc.page_content.lower() else 0
        for doc in retrieved_docs[:k]
    ]
    dcg = calculate_dcg(relevances, k)
    idcg = calculate_dcg(sorted(relevances, reverse=True), k)
    return dcg / idcg if idcg > 0 else 0.0


def evaluate_retrieval(test: TestQuestion, chunks: list[Result], k: int = 10) -> dict:
    """Evaluate retrieval quality for a test question given pre-fetched chunks."""
    mrr_scores = [calculate_mrr(kw, chunks) for kw in test.keywords]
    ndcg_scores = [calculate_ndcg(kw, chunks, k) for kw in test.keywords]
    keywords_found = sum(1 for s in mrr_scores if s > 0)
    return {
        "mrr": sum(mrr_scores) / len(mrr_scores) if mrr_scores else 0.0,
        "ndcg": sum(ndcg_scores) / len(ndcg_scores) if ndcg_scores else 0.0,
        "keyword_coverage": keywords_found / len(test.keywords) * 100 if test.keywords else 0.0,
    }

In [ ]:
# Run evaluation on a sample of test questions

tests = load_tests()
SAMPLE_SIZE = 50  # Adjust as needed (max 150)
sample_tests = tests[:SAMPLE_SIZE]

print(f"Evaluating {SAMPLE_SIZE} test questions (filtered vs unfiltered)...\n")

filtered_results = []
unfiltered_results = []

for i, test in enumerate(sample_tests):
    print(f"  [{i+1}/{SAMPLE_SIZE}] {test.question[:60]}...", end="")

    # Filtered retrieval
    f_chunks, f_types = fetch_context_with_filter(test.question)
    f_eval = evaluate_retrieval(test, f_chunks)
    filtered_results.append(f_eval)

    # Unfiltered retrieval
    u_chunks = fetch_context_no_filter(test.question)
    u_eval = evaluate_retrieval(test, u_chunks)
    unfiltered_results.append(u_eval)

    print(f" MRR: {f_eval['mrr']:.2f} vs {u_eval['mrr']:.2f}")

print("\nDone!")

In [ ]:
# Display evaluation summary

def avg(results, key):
    return sum(r[key] for r in results) / len(results) if results else 0.0

print(f"{'Metric':<22} {'Filtered':>10} {'Unfiltered':>12} {'Diff':>8}")
print("-" * 54)

for metric, label in [("mrr", "MRR"), ("ndcg", "nDCG"), ("keyword_coverage", "Keyword Coverage %")]:
    f_avg = avg(filtered_results, metric)
    u_avg = avg(unfiltered_results, metric)
    diff = f_avg - u_avg
    sign = "+" if diff >= 0 else ""
    if metric == "keyword_coverage":
        print(f"{label:<22} {f_avg:>10.1f} {u_avg:>12.1f} {sign}{diff:>7.1f}")
    else:
        print(f"{label:<22} {f_avg:>10.2f} {u_avg:>12.2f} {sign}{diff:>7.2f}")

# Count wins
wins = sum(1 for f, u in zip(filtered_results, unfiltered_results) if f["mrr"] > u["mrr"])
ties = sum(1 for f, u in zip(filtered_results, unfiltered_results) if f["mrr"] == u["mrr"])
losses = SAMPLE_SIZE - wins - ties
print(f"\nMRR: Filtered wins {wins}, ties {ties}, loses {losses} out of {SAMPLE_SIZE} questions")

## 6. 广播聊天界面

与 Insurellm 知识库进行互动聊天。界面显示：
- LLM的回答
- 哪些文档类型被分类
- 检索到的上下文块及其来源

In [ ]:
def chat(message, history):
    """Gradio chat handler."""
    # Convert Gradio history format to message list
    hist_messages = []
    for turn in history:
        hist_messages.append({"role": "user", "content": turn["content"]}) if turn["role"] == "user" else None
        hist_messages.append({"role": "assistant", "content": turn["content"]}) if turn["role"] == "assistant" else None

    answer, chunks, doc_types = answer_question(message, hist_messages)

    # Build context display
    context_parts = [f"**Classified types:** {', '.join(doc_types)}\n"]
    for i, chunk in enumerate(chunks, 1):
        source = chunk.metadata.get("source", "unknown")
        doc_type = chunk.metadata.get("type", "unknown")
        preview = chunk.page_content[:150].replace("\n", " ")
        context_parts.append(f"**{i}. [{doc_type}]** {source}\n> {preview}...\n")

    context_display = "\n".join(context_parts)
    return answer, context_display


with gr.Blocks(title="Insurellm RAG with Metadata Filtering", theme=gr.themes.Soft()) as app:
    gr.Markdown(
        "# Insurellm RAG Assistant\n"
        "*Enhanced with metadata filtering for smarter retrieval*"
    )

    with gr.Row():
        with gr.Column(scale=2):
            chatbot = gr.Chatbot(height=500, type="messages")
            msg = gr.Textbox(
                placeholder="Ask about Insurellm products, employees, contracts, or the company...",
                label="Your question",
            )
        with gr.Column(scale=1):
            context_box = gr.Markdown(
                label="Retrieved Context",
                value="*Context will appear here after your first question.*",
            )

    def respond(message, chat_history):
        answer, context_display = chat(message, chat_history)
        chat_history.append({"role": "user", "content": message})
        chat_history.append({"role": "assistant", "content": answer})
        return "", chat_history, context_display

    msg.submit(respond, [msg, chatbot], [msg, chatbot, context_box])

    gr.Examples(
        examples=[
            "What is James Wilson's salary?",
            "What is Carllm's Enterprise pricing?",
            "When was Insurellm founded and by whom?",
            "How many Bizllm contracts has Insurellm secured?",
        ],
        inputs=msg,
    )

In [ ]:
app.launch()